# Running freeCAM end to end

The CAM atmosphere of iCESM1.3.1, on 512 MPI ranks, with Python owning the
workflow: the order of physics, the clock, the coupling, and the rank-local
state.  The numbers stay Fortran's -- the original routines are called
through generated adapters -- but what runs, in what order, and what runs
beside it is decided here.

Four cells: what this checkout resolves to, what the model will run, running
it, and what came out.  Every path comes from `site.env` at the repository
root; see the README's *Site configuration*.  Nothing below names a user or
an allocation.

## 0. Does this checkout have what it needs?

A clone cannot bring the three things a 512-rank run needs: Derecho itself,
the native image (compiled in place from the pinned submodule, so it cannot
be copied between checkouts), and a configured CESM case with one completed
run.  This says which are present, and for anything absent, what produces
it -- before a queue slot is spent finding out.

In [ ]:
import os, sys
from pathlib import Path
import numpy as np

REPO = Path.cwd()
while REPO != REPO.parent and not (REPO / 'pyproject.toml').is_file():
    REPO = REPO.parent
sys.path.insert(0, str(REPO / 'src'))
from freecam import site

where = site.resolved(repo=REPO)
for name in ('account', 'queue', 'scratch', 'reference case', 'reference run'):
    print(f'{name:<16}{where[name]}')
print()

checks = site.preflight(repo=REPO)
for check in checks:
    print(f'  {check}')
absent = [check for check in checks if not check.ok]
assert not absent, (
    '\n\nnot ready.  What is missing, and what produces it:\n  '
    + '\n  '.join(f'{check.name}: {check.produced_by}' for check in absent)
    + '\n\nSee the README, section "Site configuration".')

## 1. What the model will run

No MPI yet.  A case is a workflow, and a workflow can be inspected and
changed before anything starts.  Here a Python process is inserted into the
middle of CAM's physics -- an ordinary class with a `run` method, sharing
the model's own state arrays.

In [ ]:
import freecam as fc

STEPS = int(os.environ.get('PYCAM_STEPS', 4))


class Cooling(fc.Physics):
    """A visible tendency, so the run shows whether Python reached the state."""

    name = 'python_cooling'

    def run(self, state, context):
        state.T -= 1.0e-4 * context.timestep_seconds        # 0.18 K a step


def with_cooling(default):
    workflow = default.copy()
    workflow.insert_after('dry_adjustment', Cooling())
    return workflow


case = fc.CaseConfig(
    name='PI-atm-python-cooling',
    description='PI-atm with one Python tendency inside CAM physics',
    forcing='1850 fixed preindustrial with the online CESM surface system',
    make_atm=lambda: fc.FreeCAM(workflow=with_cooling),
)

# Constructing a Driver starts nothing: preview() compiles the workflow
# and reads it back, with no PBS and no MPI.
plan = fc.Driver(case=case, nsteps=STEPS).preview()
names = [action.name for action in plan]
print(f'{len(plan)} actions in a step')
if Cooling.name in names:
    position = names.index(Cooling.name)
    print(f'  {Cooling.name} is action {position + 1}, '
          f'phase {plan[position].phase}, after {names[position - 1]}')
else:
    print(f'  {Cooling.name} is not among them')

## 2. Run it

`initialize()` launches the model: on a Derecho login node it submits a PBS
job for 512 ranks -- four nodes, charged to the account printed above -- and
waits for the workers to connect, so the first call takes as long as the
queue does.  The session then stays live: `run()` advances it, and the state
between steps is ordinary NumPy.

`cpudev` allows one allocation per user at a time, which is why `close()` at
the end matters.

In [ ]:
driver = fc.Driver(case=case, nsteps=STEPS, history_every=1, restart_every='end')
print(f'account {driver.account} (from {driver.account_source}), '
      f'queue {driver.queue}, {STEPS} steps')

driver.close()                      # release a model left over from a rerun
driver.initialize()
print('run directory:', driver.run_dir)

before = driver.cam.state.T.stats(rank='global')
result = driver.run(steps=STEPS, progress=True)
after = driver.cam.state.T.stats(rank='global')
print(f"\nglobal mean T  {before['mean']:.4f} -> {after['mean']:.4f} K "
      f"({after['mean'] - before['mean']:+.4f} over {STEPS} steps)")

## 3. Did Python actually reach the state?

Worth asking rather than assuming.  A tendency inserted into the middle of
CAM's physics is followed by convection, cloud microphysics, radiation and
the dynamical core, all of which respond to it, so the global mean after a
few steps is not evidence either way.

The action trace is: it is the model's own record of what ran, in order,
kept by the Python driver as it goes.

In [ ]:
# close() in a finally: an allocation left held is one nobody else can
# have, and cpudev allows one per user.
try:
    records = result.trace
    mine = [r for r in records if r.get('name') == Cooling.name]
    print(f'{result.actions} actions over {STEPS} steps, {len(records)} retained')
    if mine:
        print(f'  {Cooling.name} ran {len(mine)} times'
              f"{' -- once a step, as inserted' if len(mine) == STEPS else ''}")
        print(f'  first at model step {mine[0]["model_step"]}, '
              f'phase {mine[0]["phase"]}')
    else:
        # Say what the records do carry rather than assert a negative.
        print(f'  no record named {Cooling.name!r}; a record looks like '
              f'{dict(records[0]) if records else "(none kept)"}')
finally:
    driver.close()
    print('closed; the allocation is released')

## Where to go from here

* `examples/kernel_surrogate.ipynb` -- replacing a numerical kernel with a
  trained network, and what that does to the run.
* `examples/generate_training_data.ipynb` -- producing the data such a
  network is trained on.
* `validation/jobs/` -- the 512-rank, 50-step gates, submitted with
  `validation/jobs/submit.sh`.  Every numeric change is bit-for-bit against
  the pinned iCESM reference before it is believed.